# Imputation and Synthetic Missingness

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
import pandas as pd
import numpy as np
import datetime as dt
from src.imputation import impute_ts
from src.missingness import simulate_missingness, create_mask

In [3]:
df = pd.read_csv(PROJECT_ROOT / "data" / "ETTm1.csv", parse_dates=["date"], index_col="date")

In [4]:
train = df.iloc[:1000]

# Pipeline

### Split data

`train, test = ...`

**Must normalise train to 0 1 scale so that errors can be aggregated**

### Simulate missingness

Create an empty dictionary and then iterate over the 10 loops, mechanisms, and missingness levels.

In [5]:
missing_dict = {}
mask_dict = {}

for i in range(10):

    for mech in ["mar", "mcar", "mnar"]:

        for n in [10, 30, 50]:
    
            key = mech + str(n)

            if key not in mask_dict:
                mask_dict[key] = create_mask(df=train, mechanism=mech, n=n, random_seed=42)

            if key not in missing_dict:
                missing_dict[key] = []

            missing_dict[key].append(simulate_missingness(df=train, mechanism=mech, n=n, random_seed=42))

`missing_dict` will be of format `{mcar10 : [df1, df2, ... , df10], mcar30 : [df1, ...], ... , mnar50 : [df1, ... , df10]}`.

`mask_dict` will be of format `{mcar10 : mask, mcar30 : mask, ... , mnar50 : mask}`.

### Imputation

Iterate over the models and missing_dict dataframes.

In [6]:
imputed_data = {}

for model in ["locf", "ks", "arima", "lstm"]:

    for mech_prop, df_list in missing_dict.items():
        
        if model not in imputed_data:
            imputed_data[model] = {}

        imputed_data[model][mech_prop] = []

        for df in df_list:
            imputed_data[model][mech_prop].append(impute_ts(df, model))

Dictionary returned will be of format `{locf : {mcar10 : [df1, ..., df10], mcar30 : [df1, ..., df10], ..., mnar50 : [df1, ..., df10]}, ks : {}, ..., lstm : {}}`.

### Reconstruction evaluation

Need to store errors in a list, from which rmse, mape, bias and variance can be calculated for each model-mechanism-proportion combination.

